In [2]:
from torch.utils.tensorboard import SummaryWriter

In [1]:
import json, glob, math, time, copy, os
import torch, torch.nn.functional as F, torchvision, timm
from splus import SPlus

In [2]:
dev, OUT = 'cuda' if torch.cuda.is_available() else 'cpu', 'results'
BS, WARMUP, EMA_RATE = 256, 200, 0.999
os.makedirs(OUT, exist_ok=True)

In [3]:
def load(train):
    ds = torchvision.datasets.CIFAR10('./data', train=train, download=True)
    x = torch.tensor(ds.data).permute(0, 3, 1, 2).float().div(255)
    mean = torch.tensor([0.4914, 0.4822, 0.4465]).view(1, 3, 1, 1)
    std = torch.tensor([0.2470, 0.2435, 0.2616]).view(1, 3, 1, 1)
    return ((x - mean) / std).to(dev), torch.tensor(ds.targets).to(dev)

Xtr, Ytr = load(True)
Xte, Yte = load(False)

/opt/homebrew/lib/python3.11/site-packages/torchvision/datasets/cifar.py:83: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  entry = pickle.load(f, encoding="latin1")


In [4]:
def augment(x):  # random crop (pad 4) + horizontal flip, поэлементно на GPU
    B = x.size(0)
    x = torch.where((torch.rand(B, device=dev) < 0.5)[:, None, None, None], x.flip(3), x)
    xp = F.pad(x, (4, 4, 4, 4))
    i, j = torch.randint(0, 9, (B,), device=dev), torch.randint(0, 9, (B,), device=dev)
    ar = torch.arange(32, device=dev)
    rows, cols = (i[:, None] + ar)[:, :, None], (j[:, None] + ar)[:, None, :]
    return xp[torch.arange(B, device=dev)[:, None, None], :, rows, cols].permute(0, 3, 1, 2)


In [5]:
def make_model():
    if SMOKE:  # ~0.1M параметров, 16 токенов
        kw = dict(patch_size=8, embed_dim=64, depth=2, num_heads=4, mlp_ratio=2.0)
    else:      # ~4.8M параметров
        kw = dict(patch_size=4, embed_dim=256, depth=6, num_heads=4, mlp_ratio=4.0)
    return timm.models.vision_transformer.VisionTransformer(
        img_size=32, num_classes=10, **kw).to(dev)

In [6]:
CFG = {'adam': (0.0, 'cos', False), 'adamw': (0.05, 'cos', False),
       'splus_cos': (0.1, 'cos', False), 'splus_ema': (0.1, 'const', True)}
GRID = {k: [c * 10 ** (e / 3) for e in (range(-3, 4) if k.startswith('adam') else range(-4, 3))]
        for k, c in [('adam', 1e-3), ('adamw', 1e-3), ('splus_cos', 0.2), ('splus_ema', 0.2)]}


In [7]:
def build(name, model, lr, wd):
    if name.startswith('adam'):
        cls = torch.optim.Adam if name == 'adam' else torch.optim.AdamW
        opts = [cls(model.parameters(), lr=lr, weight_decay=wd)]
    else:
        head = list(model.head.parameters())
        ids = {id(p) for p in head}
        body = [p for p in model.parameters() if id(p) not in ids]
        # max_dim=0: для головы вместо собственного базиса берётся sign(momentum)
        opts = [SPlus([{'params': body}, {'params': head, 'max_dim': 0}],
                      lr=lr, weight_decay=wd)]
    for o in opts:
        for g in o.param_groups: g['base_lr'] = g['lr']
    return opts

In [8]:
def factor(step, total, sched):  # warmup + (cosine | const)
    w = min(1.0, (step + 1) / WARMUP)
    return w * (0.5 * (1 + math.cos(math.pi * step / total)) if sched == 'cos' else 1.0)


In [9]:
@torch.no_grad()
def evaluate(model):
    model.eval(); loss, correct = 0.0, 0
    for i in range(0, len(Xte), 1000):
        x, y = Xte[i:i + 1000], Yte[i:i + 1000]
        with torch.autocast('cuda', torch.bfloat16): out = model(x)
        loss += F.cross_entropy(out.float(), y, reduction='sum').item()
        correct += (out.argmax(1) == y).sum().item()
    return loss / len(Xte), correct / len(Xte)

In [ ]:
def run(name, lr, seed, epochs, stage, log_every=50):
    tag = f'{stage}_{name}_{lr:.4g}_s{seed}'
    path = f'{OUT}/{tag}.json'
    if os.path.exists(path): return
    wd, sched, use_ema = CFG[name]
    torch.manual_seed(seed)
    model = make_model()
    opts = build(name, model, lr, wd)
    writer = SummaryWriter(f'{OUT}/tb/{tag}')
    spe = len(Xtr) // BS; total = epochs * spe; step = 0
    log = dict(cfg=dict(name=name, lr=lr, wd=wd, sched=sched, seed=seed, epochs=epochs),
               train_loss=[], test_loss=[], test_acc=[])
    t0 = time.time()
    for ep in range(epochs):
        model.train(); perm = torch.randperm(len(Xtr), device=dev); run_loss = torch.zeros((), device=dev)
        for k in range(spe):
            idx = perm[k * BS:(k + 1) * BS]
            f = factor(step, total, sched)
            for o in opts:
                for g in o.param_groups: g['lr'] = g['base_lr'] * f
                o.zero_grad(set_to_none=True)
            with torch.autocast('cuda', torch.bfloat16): out = model(augment(Xtr[idx]))
            loss = F.cross_entropy(out.float(), Ytr[idx])
            loss.backward()
            for o in opts: o.step()
            run_loss += loss.detach(); step += 1
            if step % log_every == 0:  # .item() синхронизирует с GPU, поэтому не на каждом шаге
                writer.add_scalar('step/train_loss', loss.item(), step)
                writer.add_scalar('step/lr_factor', f, step)
        if use_ema: opts[0].eval()
        tl, ta = evaluate(model)
        if use_ema: opts[0].train()
        trl = run_loss.item() / spe
        log['train_loss'].append(trl); log['test_loss'].append(tl); log['test_acc'].append(ta)
        writer.add_scalar('epoch/train_loss', trl, ep)
        writer.add_scalar('epoch/test_loss', tl, ep)
        writer.add_scalar('epoch/test_acc', ta, ep)
    log['time'] = time.time() - t0
    writer.close()
    json.dump(log, open(path, 'w'))

In [11]:
def best_lr(name):
    res = []
    for f in glob.glob(f'{OUT}/sweep_{name}_*.json'):
        r = json.load(open(f)); v = r['test_loss'][-1]
        res.append((v if v == v else float('inf'), r['cfg']['lr']))  # NaN -> inf
    return min(res)[1]

def sweep(epochs=30):
    for name in CFG:
        for lr in GRID[name]: run(name, lr, seed=0, epochs=epochs, stage='sweep')

def final(epochs=300, seeds=(1, 2, 3)):
    for name in CFG:
        lr = best_lr(name)
        for s in seeds: run(name, lr, seed=s, epochs=epochs, stage='final')

In [ ]:
SMOKE = False
if SMOKE:
    OUT, WARMUP = 'results_smoke', 20                   # warmup короче: шагов на эпоху мало
    os.makedirs(OUT, exist_ok=True)
    Xtr, Ytr = Xtr[:10_000], Ytr[:10_000]               # 39 шагов на эпоху вместо 195
    Xte, Yte = Xte[:2_000],  Yte[:2_000]
    GRID = {k: v[2:5] for k, v in GRID.items()}         # 3 значения lr вокруг центра

sweep()
final()

/var/folders/sh/vd4xfx0s751452y9l4y5l2yc0000gn/T/ipykernel_7286/2856673393.py:20: UserWarning: CUDA is not available or torch_xla is imported. Disabling autocast.
  with torch.autocast('cuda', torch.bfloat16): out = model(augment(Xtr[idx]))
/var/folders/sh/vd4xfx0s751452y9l4y5l2yc0000gn/T/ipykernel_7286/2468125938.py:6: UserWarning: CUDA is not available or torch_xla is imported. Disabling autocast.
  with torch.autocast('cuda', torch.bfloat16): out = model(x)


In [ ]:
#добавить tensor board.